# 6.8 訓練窗口是在切詞嗎？


原文已由tokenizer變成ID0、1、2……，模型一次只看三個位置，怎麼辦？我們從這份ID序列取短段作為訓練題，這叫窗口。它決定每題看到多遠，沒有重新定義哪個字應合成哪個token。把token拆分與窗口長度混在一起，會讓你誤以為改context就改變了詞表。

[6.2](https://birdhackor.github.io/tiny-perceptron-vlm/6.2.html) 的合併發生在文字表示規則，[2.5](https://birdhackor.github.io/tiny-perceptron-vlm/2.5.html) 的context則是可見前文範圍。為建立C格下一token問題，還要有它們右邊的C格答案，所以從原始ID需取C+1格，再按 [1.3](https://birdhackor.github.io/tiny-perceptron-vlm/1.3.html) shift一次。C=3時取 `[0,1,2,3]`，輸入 `[0,1,2]`，答案 `[1,2,3]`。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
from tiny_perceptron.data import shifted

ids = list(range(10))
context = 3
for start in [0, 3, 6]:
    segment = ids[start : start + context + 1]
    x, y = shifted(segment)
    print("起點", start, "輸入", x.tolist(), "答案", y.tolist())
    assert len(x) == len(y) == context

輸入十個已有ID與窗口長度3，`range(10)` 建0到9的一排整數；本例它們只當容易追蹤的代號。三起點分別取0–3、3–6、6–9，共各四格；shift後輸出 `(0,1,2)→(1,2,3)`、`(3,4,5)→(4,5,6)`、`(6,7,8)→(7,8,9)`。`.tolist()` 將tensor轉回普通清單方便顯示，檢查每題確實三輸入、三答案。

相鄰窗口共用邊界token3與6，但它們提供的位置角色不同；本例所有題屬於同一份已分組訓練資料，允許這種共用。不能先從同一文件取這些高度重疊窗口，再隨機分訓練與驗證，[1.2](https://birdhackor.github.io/tiny-perceptron-vlm/1.2.html) 已說明原因。也不能跨文件邊界直接連成一個「上一文件末尾預測下一文件開頭」目標，除非明確設計並標記這種邊界。

資料末尾還要處理長度不足。若剩下的段只有C格，shift只能給C-1道題；我們的檢查會發現與目標C不合。可以丟掉尾段，或另明示補齊與忽略答案規則，但不應悄悄當作足長題。第7章會處理不同長短batch，當前用剛好可取的起點。

練習只把context改成4，保留三個起點，先預測起點6只有 `[6,7,8,9]` 四個原ID，shift後只有三道題，最後檢查失敗。執行核對，再把起點列表改成 `[0,4]`，兩次都可取五格、建立四題。這次沒有改tokenizer，只改每題資料取段的契約。
